# Lezione 06 — Analisi del sentimento classica

**Corso:** NLP Classico con Python · **Blocco 2:** Rappresentare e classificare
**Tempo stimato:** 100 minuti · **Capitolo del manuale:** `manual/06_sentimiento_clasico_IT.md`

## Cosa vedrai in questa lezione

- Un **analizzatore lessicale** del sentimento: un piccolo **lessico di polarità** (elenco di parole con un valore da −2 a +2) e una somma di punteggi.
- **Negazione** (una finestra di parole) e **intensificatori** (`molto`, `poco`): cosa sistemano e dove restano corti.
- Un classificatore **supervisionato** (TF-IDF + regressione logistica) e il suo **confronto con il lessico sullo stesso corpus e sulla stessa partizione**.
- Perché il **sacco di parole perde la negazione**, con un esperimento guidato.
- Il **sarcasmo e l'ironia** come limite, e l'**analisi degli errori per categorie** (invece di un'unica percentuale).

## Prerequisiti

- Sacco di parole e TF-IDF (lezione 4) e classificazione con regressione logistica, partizione addestramento/test e metriche (lezione 5).
- Librerie: `numpy`, `pandas`, `matplotlib`, `scikit-learn`. La cella seguente installa solo ciò che manca (funziona allo stesso modo in Google Colab).
- La cartella `datos/` del corso: `resenas_es.csv`, `resenas_it.csv` e i due lessici `lexico_polaridad_es.csv`, `lexico_polaridad_it.csv`. In Colab, caricala dal pannello dei file.

> **Avvertenza sulle cifre:** il corpus è sintetico e piccolo (300 recensioni per lingua) e il lessico è un **lessico minuscolo scritto per il corso**. Tutte le cifre sono un *esempio illustrativo*: non dicono che un metodo sia migliore di un altro in generale, e **non promettiamo che il lessico batta il modello supervisionato: misuriamo ciò che esce**.

> **Come usare questo notebook:** esegui le celle in ordine. È una lezione densa (100 minuti): le sezioni 6 e 8 si possono lasciare come lettura se manca il tempo.

---
## Installazione

Installa solo ciò che manca. In un ambiente con tutto installato non fa nulla.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("scipy", "scipy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
print("Librerie pronte.")

---
## Cella 0 — Setup

Importiamo le librerie, fissiamo il **seme**, definiamo `cargar_datos` e leggiamo le recensioni. Facciamo **una sola partizione** addestramento/test (70/30, stratificata per etichetta) e la useremo per tutto: il lessico e il modello supervisionato vengono valutati sulle **stesse 90 recensioni di test** di ciascuna lingua.

In [ ]:
import os
import re
import random
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

random.seed(42)                          # seme fisso: tutti otteniamo gli stessi risultati
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

IDIOMAS = ("spa", "ita")
ARCHIVO_RESENAS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}
ARCHIVO_LEXICO = {"spa": "lexico_polaridad_es.csv", "ita": "lexico_polaridad_it.csv"}
CLASES = ["negativo", "neutro", "positivo"]

resenas = {i: pd.read_csv(cargar_datos(a)) for i, a in ARCHIVO_RESENAS.items()}

# Un'unica partizione addestramento/test per TUTTO il notebook (la stessa per il lessico e per il modello)
train, test = {}, {}
for i, df in resenas.items():
    train[i], test[i] = train_test_split(df, test_size=0.3, stratify=df["etiqueta"], random_state=42)
    print(i, "totale", len(df), "· train", len(train[i]), "· test", len(test[i]))
print("Dati e librerie caricati correttamente.")

---
## 1. Il corpus: recensioni con tre classi e casi difficili

Le recensioni sono etichettate `positivo`, `negativo` o `neutro`, con le classi bilanciate. La colonna `dificil` segna (≈ 15 %) i casi con negazione, ironia o giudizi misti che **abbiamo scritto apposta** per mettere alla prova i metodi. Guardiamo solo l'**addestramento**: il test si tiene da parte per misurare.

In [ ]:
for idioma in IDIOMAS:
    df = train[idioma]
    print(f"=== {idioma} ===")
    print(pd.crosstab(df["etiqueta"], df["dificil"], margins=True))
    print(df.sort_values("id").groupby("etiqueta").head(1)[["id", "etiqueta", "dificil", "texto_limpio"]].to_string(index=False))
    print()

**Cosa osservi?** Le tre classi sono bilanciate e i casi `dificil=1` sono distribuiti tra di esse. Nota che la classe `neutro` **non è l'assenza di opinione**: comprende recensioni con lodi e lamentele insieme («carino ma con poche opzioni»). Questo peserà molto più avanti.

---
## 2. Il lessico di polarità: cosa contiene e come è stato compilato

`datos/lexico_polaridad_es.csv` e `datos/lexico_polaridad_it.csv` sono **elenchi scritti per il corso** (≈ 160 voci per lingua, con valori −2, −1, +1, +2). Sono stati compilati **a memoria, a partire da vocabolario generale di opinione, prima di misurare qualsiasi cosa sul corpus**, e non sono copiati da nessun lessico esistente né ritoccati dopo guardando gli errori. Contengono solo parole con carica; i negatori e gli intensificatori sono nel codice.

È trasparente (puoi leggerlo per intero) e non richiede dati etichettati, ma è **minuscolo**: un lessico serio ha migliaia di voci.

In [ ]:
# datos-es
NEGADORES = {"spa": {"no", "ni", "nunca", "jamas", "tampoco", "sin", "nada", "ningun", "ninguno", "ninguna"},
             "ita": {"non", "mai", "senza", "nessun", "nessuno", "nessuna", "nemmeno", "neppure", "niente", "nulla"}}
INTENSIFICADORES = {"spa": {"muy": 1.5, "super": 1.5, "tremendamente": 1.5, "realmente": 1.5, "totalmente": 1.5,
                            "bastante": 1.25, "demasiado": 1.25, "poco": 0.5, "algo": 0.5},
                    "ita": {"molto": 1.5, "super": 1.5, "davvero": 1.5, "veramente": 1.5, "assolutamente": 1.5,
                            "estremamente": 1.5, "troppo": 1.25, "abbastanza": 1.25, "poco": 0.5, "alquanto": 0.75}}
CORTES = {"spa": {"pero", "aunque", "sino"}, "ita": {"ma", "pero", "sebbene", "tuttavia"}}   # le avversative tagliano la finestra

In [ ]:
def normalizar(texto):
    """Minuscolo e senza accenti: così il lessico funziona anche con il testo grezzo che ha perso gli accenti."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

def tokenizar(texto):
    """Parole e segni di punteggiatura (i segni delimitano la finestra della negazione)."""
    return re.findall(r"\w+|[.,;:!?()]", normalizar(texto))

def cargar_lexico(idioma):
    tabla = pd.read_csv(cargar_datos(ARCHIVO_LEXICO[idioma]))
    return {normalizar(p): float(v) for p, v in zip(tabla["palabra"], tabla["polaridad"])}

def analizar(texto, lexico, idioma, ventana=3, negacion=True, intensificadores=True):
    """Restituisce (punteggio totale, lista dei contributi (parola, valore base, valore finale))."""
    tokens = tokenizar(texto)
    total, aportes = 0.0, []
    for i, palabra in enumerate(tokens):
        if palabra not in lexico:
            continue
        valor = lexico[palabra]
        multiplicador, invertir = 1.0, False
        for j in range(i - 1, max(i - ventana, 0) - 1, -1):    # all'indietro, fino a `ventana` token
            anterior = tokens[j]
            if not anterior[0].isalnum() or anterior in CORTES[idioma]:
                break                                          # punteggiatura o «ma»: fine della proposizione
            if negacion and anterior in NEGADORES[idioma]:
                invertir = True                                # più negatori di seguito invertono UNA sola volta
            elif intensificadores and anterior in INTENSIFICADORES[idioma]:
                multiplicador *= INTENSIFICADORES[idioma][anterior]
        final = valor * multiplicador * (-1 if invertir else 1)
        aportes.append((palabra, valor, final))
        total += final
    return total, aportes

def puntuar(texto, lexico, idioma, **opciones):
    return analizar(texto, lexico, idioma, **opciones)[0]

LEXICO = {idioma: cargar_lexico(idioma) for idioma in IDIOMAS}

In [ ]:
for idioma in IDIOMAS:
    tabla = pd.read_csv(cargar_datos(ARCHIVO_LEXICO[idioma]))
    print(f"=== {idioma}: {len(tabla)} voci ===")
    print(tabla["polaridad"].value_counts().sort_index().to_dict())
    print(tabla.groupby("polaridad")["palabra"].apply(lambda s: ", ".join(list(s)[:6])).to_string())
    print()

### Il lessico «conosce» il corpus? Un controllo di sospetto

Se il lessico fosse stato costruito **copiando parole dal corpus** (e soprattutto dal test), quasi tutte le sue voci comparirebbero nelle recensioni e l'accuratezza sul test sarebbe sospettosamente alta. Controlliamo tre cose: quante voci **non compaiono in nessuna recensione**, quante compaiono **solo nel test**, e se l'accuratezza su addestramento e su test è simile (se fosse ritoccato sul test, lì sarebbe migliore).

In [ ]:
def vocabulario(df, columna="texto_limpio"):
    return {t for texto in df[columna] for t in re.findall(r"\w+", normalizar(texto))}

for idioma in IDIOMAS:
    lex = LEXICO[idioma]
    v_train, v_test = vocabulario(train[idioma]), vocabulario(test[idioma])
    en_train = [p for p in lex if p in v_train]
    solo_test = [p for p in lex if p in v_test and p not in v_train]
    en_ninguna = [p for p in lex if p not in v_train and p not in v_test]
    print(f"{idioma}: {len(lex)} voci · nell'addestramento: {len(en_train)} · solo nel test: {len(solo_test)} · in nessuna recensione: {len(en_ninguna)}")
    for nombre, df in (("train", train[idioma]), ("test", test[idioma])):
        con_palabra = np.mean([bool(analizar(t, lex, idioma)[1]) for t in df["texto"]])
        print(f"   {nombre}: recensioni con almeno una parola del lessico = {con_palabra:.3f}")

**Cosa osservi?** Una parte grande del lessico (più di un terzo in spagnolo e più della metà in italiano) **non compare in nessuna recensione**: non è stato costruito a partire dal corpus. Le voci che compaiono solo nel test sono ≈ 10 per lingua, e la **copertura** (recensioni con almeno una parola del lessico) si muove tra il 52 % e il 74 % a seconda della lingua e della partizione, **senza una direzione comune**: maggiore nel test in spagnolo, minore in italiano (con 90 recensioni, quelle differenze sono rumore di campionamento). Questo non *dimostra* che non ci sia distorsione, ma è coerente con un lessico generale; l'accuratezza su addestramento e su test si confronta nella sezione 4.

---
## 3. L'analizzatore passo per passo

`analizar` scorre i token; per ogni parola del lessico guarda **all'indietro fino a `ventana` token** dentro la stessa proposizione (la punteggiatura e `pero`/`ma` la interrompono):

- se trova un **negatore** (`non`, `mai`, `senza`…), **inverte** il segno (una volta sola anche se ce ne sono più di uno);
- se trova un **intensificatore**, moltiplica: `molto` ×1,5 · `poco` ×0,5.

Lo proviamo con frasi piccole il cui punteggio si calcola **a mano** (con `ventana=3`).

In [ ]:
# datos-es
FRASES = {
    "spa": ["el servicio es bueno",
            "el servicio no es bueno",
            "el servicio es muy bueno",
            "no es nada malo",
            "ni bueno ni malo",
            "bueno pero caro",
            "sin problemas",
            "no llegó tarde"],
    "ita": ["il servizio è buono",
            "il servizio non è buono",
            "il servizio è molto buono",
            "non è per niente male",
            "né buono né cattivo",
            "buono ma caro",
            "senza problemi",
            "non è arrivato in ritardo"],
}

In [ ]:
ESPERADAS = [1.0, -1.0, 1.5, 1.0, 0.0, 0.0, 1.0, 1.0]     # calcolate a mano
ESPERADAS_ITA = [1.0, -1.0, 1.5, 1.0, 0.0, 0.0, 1.0, -1.0]  # «non è arrivato in ritardo»: il negatore resta a 4 token

for idioma, esperadas in (("spa", ESPERADAS), ("ita", ESPERADAS_ITA)):
    filas = []
    for frase, esperada in zip(FRASES[idioma], esperadas):
        con = puntuar(frase, LEXICO[idioma], idioma)
        sin = puntuar(frase, LEXICO[idioma], idioma, negacion=False)
        assert con == esperada, (frase, con, esperada)
        filas.append({"frase": frase, "senza negazione": sin, "con negazione": con})
    print(pd.DataFrame(filas).to_string(index=False))
    print()
print("Tutti i punteggi coincidono con il calcolo a mano ✔")

**Cosa osservi?** Senza negazione, «il servizio non è buono» ottiene **+1** (lo stesso segno di «è buono»); con la finestra passa a **−1**. «Né buono né cattivo» dà 0 e «buono ma caro» dà 0 (`ma` interrompe la finestra). E c'è un errore istruttivo: in italiano, «non è arrivato in ritardo» resta a **−1** perché `non` è a **quattro** token da `ritardo` e la finestra è 3. Con `ventana=4` si sistemerebbe, ma una finestra più larga comincia a invertire parole che non c'entrano: **non esiste un valore perfetto**.

### Trasparenza: perché una recensione riceve il suo punteggio

Un vantaggio reale del lessico: si può **spiegare** ogni decisione parola per parola.

In [ ]:
for idioma in IDIOMAS:
    candidatas = train[idioma][train[idioma]["dificil"] == 0].sort_values("id")
    fila = next(f for _, f in candidatas.iterrows() if len(analizar(f["texto_limpio"], LEXICO[idioma], idioma)[1]) >= 3)
    total, aportes = analizar(fila["texto_limpio"], LEXICO[idioma], idioma)
    print(f"[{idioma}] ({fila['etiqueta']}) {fila['texto_limpio']}")
    for palabra, base, final in aportes:
        print(f"   {palabra:<14} base {base:+.1f}  →  {final:+.2f}")
    print(f"   TOTALE = {total:+.2f}\n")

---
## 4. Dal punteggio alla classe: soglia e finestra, scelte con l'addestramento

Il punteggio è un numero; per ottenere `positivo` / `neutro` / `negativo` serve una **soglia**: sopra, positivo; sotto −soglia, negativo; in mezzo, neutro. È l'**unico parametro che regoliamo**, e lo facciamo **solo con l'addestramento**. Ne approfittiamo per vedere quanto influisce la finestra della negazione (sul testo grezzo, come per il modello supervisionato).

In [ ]:
def clasificar(puntuacion, umbral):
    """Punteggio → classe: sopra la soglia, positivo; sotto −soglia, negativo; in mezzo, neutro."""
    if puntuacion > umbral:
        return "positivo"
    if puntuacion < -umbral:
        return "negativo"
    return "neutro"

REJILLA_UMBRAL = (0, 0.25, 0.5, 0.75, 1.0)

def mejor_umbral(puntuaciones, y, rejilla=REJILLA_UMBRAL):
    """Soglia con la maggiore accuratezza su (punteggi, y); a parità, la minore."""
    mejor = max(rejilla, key=lambda u: (accuracy_score(y, [clasificar(p, u) for p in puntuaciones]), -u))
    return mejor, accuracy_score(y, [clasificar(p, mejor) for p in puntuaciones])

def modelo_supervisado(ngramas=(1, 2)):
    return make_pipeline(TfidfVectorizer(ngram_range=ngramas, sublinear_tf=True),
                         LogisticRegression(max_iter=1000))

In [ ]:
CONFIGS = {
    "senza negazione": dict(negacion=False),
    "finestra 1": dict(ventana=1),
    "finestra 2": dict(ventana=2),
    "finestra 3 (predefinita)": dict(ventana=3),
    "finestra 4": dict(ventana=4),
    "finestra 5": dict(ventana=5),
    "senza intensificatori": dict(intensificadores=False),
}
filas = []
for nombre, opciones in CONFIGS.items():
    fila = {"configurazione": nombre}
    for idioma in IDIOMAS:
        puntos = [puntuar(t, LEXICO[idioma], idioma, **opciones) for t in train[idioma]["texto"]]
        umbral, acierto = mejor_umbral(puntos, train[idioma]["etiqueta"])
        fila[f"{idioma} soglia"] = umbral
        fila[f"{idioma} accuratezza (train)"] = round(acierto, 3)
    filas.append(fila)
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** Le differenze tra le configurazioni sono **piccole**: con 210 recensioni di addestramento, un punto di accuratezza sono ≈ 2 recensioni. Non leggere nella tabella una «finestra ottimale»: è rumore entro ciò che questo campione permette di distinguere. Manteniamo `ventana=3` per principio (la finestra predefinita del capitolo) e fissiamo la **soglia con l'addestramento**.

In [ ]:
UMBRAL = {}
for idioma in IDIOMAS:
    puntos = [puntuar(t, LEXICO[idioma], idioma) for t in train[idioma]["texto"]]
    UMBRAL[idioma], acierto_train = mejor_umbral(puntos, train[idioma]["etiqueta"])
    puntos_test = [puntuar(t, LEXICO[idioma], idioma) for t in test[idioma]["texto"]]
    acierto_test = accuracy_score(test[idioma]["etiqueta"], [clasificar(p, UMBRAL[idioma]) for p in puntos_test])
    print(f"{idioma}: soglia scelta = {UMBRAL[idioma]} · accuratezza del lessico su train = {acierto_train:.3f} · su test = {acierto_test:.3f}")

Se il lessico fosse ritoccato sul test, l'accuratezza sul test sarebbe chiaramente **maggiore** di quella sull'addestramento. Non è ciò che vediamo (con 90 recensioni di test, pochi punti di differenza sono rumore), coerente con la compilazione descritta nella sezione 2.

---
## 5. Il classificatore supervisionato e il confronto sulla stessa partizione

Addestriamo `TfidfVectorizer` (unigrammi e bigrammi, `sublinear_tf`) + `LogisticRegression` **con lo stesso addestramento** e lo misuriamo sullo **stesso test**, sullo stesso testo grezzo. Confrontiamo accuratezza, F1 macro e l'accuratezza sui casi `dificil` (con pochissime recensioni: si indica `n`).

In [ ]:
def resumen(y, pred, dificil):
    y, pred, dificil = np.asarray(y), np.asarray(pred), np.asarray(dificil)
    return {"accuratezza": round(accuracy_score(y, pred), 3),
            "F1 macro": round(f1_score(y, pred, average="macro"), 3),
            "accuratezza facili": round(accuracy_score(y[dificil == 0], pred[dificil == 0]), 3),
            "accuratezza difficili": round(accuracy_score(y[dificil == 1], pred[dificil == 1]), 3),
            "n difficili": int((dificil == 1).sum())}

PRUEBA = {}
filas = []
for idioma in IDIOMAS:
    y = test[idioma]["etiqueta"]
    pred_lex = [clasificar(puntuar(t, LEXICO[idioma], idioma), UMBRAL[idioma]) for t in test[idioma]["texto"]]
    modelo = modelo_supervisado().fit(train[idioma]["texto"], train[idioma]["etiqueta"])
    pred_lr = modelo.predict(test[idioma]["texto"])
    PRUEBA[idioma] = {"lex": pred_lex, "lr": pred_lr}
    for nombre, pred in (("lessico", pred_lex), ("TF-IDF + LR", pred_lr)):
        filas.append({"lingua": idioma, "metodo": nombre, **resumen(y, pred, test[idioma]["dificil"])})
TABLA = pd.DataFrame(filas)
print(TABLA.to_string(index=False))

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for eje, idioma in zip(ejes, IDIOMAS):
    sub = TABLA[TABLA["lingua"] == idioma]
    eje.bar(sub["metodo"], sub["accuratezza"], color=["#8fb4d9", "#e9a15b"])
    eje.axhline(1 / 3, color="gray", linestyle="--", linewidth=1)
    eje.set_title(f"{idioma} (n = {len(test[idioma])})")
    eje.set_ylim(0, 1)
ejes[0].set_ylabel("accuratezza sul test")
plt.suptitle("Stessa partizione e stesso testo (la linea grigia è il caso: 1/3)")
plt.tight_layout()
plt.show()

**Cosa osservi?** Confronta le due righe di ciascuna lingua. Con **90 recensioni di test**, una recensione vale ≈ 1,1 punti di accuratezza e le differenze di pochi punti **non sono conclusive**. E nei casi `dificil` ci sono una decina di recensioni: lì le cifre sono ancora più instabili. Nella sezione seguente ripetiamo la partizione molte volte per non dipendere da una sola.

---
## 6. Dipende dalla partizione? Venti partizioni diverse

Ripetiamo 20 volte la partizione 70/30 (semi da 0 a 19): in ciascuna la soglia del lessico si sceglie con **il suo** addestramento e il modello supervisionato si addestra con **il suo** addestramento. È un controllo di robustezza, non una prova statistica: i 20 test si sovrappongono.

In [ ]:
resultados = {}
for idioma in IDIOMAS:
    df = resenas[idioma].reset_index(drop=True)
    puntos = np.array([puntuar(t, LEXICO[idioma], idioma) for t in df["texto"]])
    a_lex, a_lr = [], []
    for semilla in range(20):
        idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.3, stratify=df["etiqueta"], random_state=semilla)
        y = df["etiqueta"].to_numpy()
        umbral, _ = mejor_umbral(puntos[idx_tr], y[idx_tr])
        a_lex.append(accuracy_score(y[idx_te], [clasificar(p, umbral) for p in puntos[idx_te]]))
        modelo = modelo_supervisado().fit(df["texto"].iloc[idx_tr], y[idx_tr])
        a_lr.append(accuracy_score(y[idx_te], modelo.predict(df["texto"].iloc[idx_te])))
    resultados[idioma] = (np.array(a_lex), np.array(a_lr))
    lex, lr = resultados[idioma]
    print(f"{idioma}: lessico {lex.mean():.3f} ± {lex.std():.3f} · TF-IDF + LR {lr.mean():.3f} ± {lr.std():.3f} · il modello supera il lessico in {int((lr > lex).sum())}/20 partizioni")

**Cosa osservi?** La media e la deviazione standard (± tra partizioni) danno un'idea di quanto si muove la cifra a seconda della partizione. Confronta la differenza tra i metodi con quella deviazione: se è dello stesso ordine, bisogna essere prudenti nel concludere, e parliamo solo di **questo corpus sintetico**, non di recensioni reali.

---
## 7. Esperimento guidato: perché il sacco di parole perde la negazione

Un sacco di parole **non ha ordine**: «il servizio non è buono» e «il servizio è buono» si distinguono solo per una parola in più, `non`. Prima lo misuriamo a mano con il **coseno** tra i due vettori di conteggi: condividono 4 parole e la seconda ne aggiunge una.

In [ ]:
# datos-es
PARES_POS = {   # aggettivo positivo: ci si aspetta che la negazione ABBASSI il sentimento
    "spa": [("el servicio es bueno", "el servicio no es bueno"),
            ("la comida es buena", "la comida no es buena"),
            ("el hotel es limpio", "el hotel no es limpio"),
            ("la aplicación es rápida", "la aplicación no es rápida"),
            ("el envío es puntual", "el envío no es puntual"),
            ("la habitación es cómoda", "la habitación no es cómoda")],
    "ita": [("il servizio è buono", "il servizio non è buono"),
            ("il cibo è buono", "il cibo non è buono"),
            ("l'hotel è pulito", "l'hotel non è pulito"),
            ("l'app è veloce", "l'app non è veloce"),
            ("la consegna è puntuale", "la consegna non è puntuale"),
            ("la camera è comoda", "la camera non è comoda")],
}
PARES_NEG = {   # aggettivo negativo: ci si aspetta che la negazione ALZI il sentimento
    "spa": [("el servicio es malo", "el servicio no es malo"),
            ("la comida es mala", "la comida no es mala"),
            ("el hotel es sucio", "el hotel no es sucio"),
            ("la aplicación es lenta", "la aplicación no es lenta"),
            ("el envío es caro", "el envío no es caro"),
            ("el altavoz está roto", "el altavoz no está roto")],
    "ita": [("il servizio è cattivo", "il servizio non è cattivo"),
            ("il cibo è cattivo", "il cibo non è cattivo"),
            ("l'hotel è sporco", "l'hotel non è sporco"),
            ("l'app è lenta", "l'app non è lenta"),
            ("la consegna è cara", "la consegna non è cara"),
            ("l'altoparlante è rotto", "l'altoparlante non è rotto")],
}

In [ ]:
def coseno_bolsa(a, b):
    ta, tb = re.findall(r"\w+", normalizar(a)), re.findall(r"\w+", normalizar(b))
    vocab = sorted(set(ta) | set(tb))
    va, vb = np.array([ta.count(w) for w in vocab]), np.array([tb.count(w) for w in vocab])
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))

for idioma in IDIOMAS:
    base, negada = PARES_POS[idioma][0]
    cos = coseno_bolsa(base, negada)
    assert abs(cos - 4 / (2 * 5 ** 0.5)) < 1e-12          # 4 parole in comune; norme 2 e √5
    print(f"{idioma}: «{base}» contro «{negada}» → coseno = {cos:.3f}")

I due vettori hanno un coseno di 0,894: **quasi identici**, anche se il senso è opposto. Ora confrontiamo quattro approcci (misuriamo la variazione del **margine** positivo − negativo aggiungendo `non`) su dodici coppie minime (sei con un aggettivo positivo, «X è buono» / «X non è buono», e sei con uno negativo, «X è cattivo» / «X non è cattivo»), con i modelli addestrati sullo **stesso addestramento** di sempre: (a) TF-IDF con **unigrammi**, (b) con **unigrammi e bigrammi**, (c) unigrammi e bigrammi con **marcature di negazione** (`NEG_` davanti alle parole che seguono un negatore, dentro la finestra) e (d) il **lessico con finestra**.

In [ ]:
def marcar_negacion(texto, idioma, ventana=3):
    """Prefissa NEG_ alle parole che seguono un negatore (fino a `ventana` o fino a un segno di punteggiatura)."""
    tokens = tokenizar(texto)
    salida, restantes = [], 0
    for t in tokens:
        if not t[0].isalnum() or t in CORTES[idioma]:
            restantes = 0
            salida.append(t)
        elif t in NEGADORES[idioma]:
            restantes = ventana
            salida.append(t)
        elif restantes > 0:
            salida.append("NEG_" + t)
            restantes -= 1
        else:
            salida.append(t)
    return " ".join(salida)

def modelo_con_marcas(idioma):
    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, lowercase=False, token_pattern=r"\S+",
                          preprocessor=lambda t: marcar_negacion(t, idioma))
    return make_pipeline(vec, LogisticRegression(max_iter=1000))

In [ ]:
def margen(modelo, frases):
    """Margine positivo − negativo della funzione di decisione (maggiore = più positivo)."""
    clases = list(modelo.classes_)
    d = modelo.decision_function(frases)
    return d[:, clases.index("positivo")] - d[:, clases.index("negativo")]

filas = []
for idioma in IDIOMAS:
    modelos = {"(a) unigrammi": modelo_supervisado((1, 1)),
               "(b) uni + bigrammi": modelo_supervisado((1, 2)),
               "(c) con marcature NEG_": modelo_con_marcas(idioma)}
    for nombre, modelo in modelos.items():
        modelo.fit(train[idioma]["texto"], train[idioma]["etiqueta"])
        fila = {"lingua": idioma, "approccio": nombre}
        for grupo, pares, signo in (("POS", PARES_POS, -1), ("NEG", PARES_NEG, +1)):
            cambio = margen(modelo, [n for _, n in pares[idioma]]) - margen(modelo, [b for b, _ in pares[idioma]])
            fila[f"{grupo}: coppie ben orientate"] = int((signo * cambio > 0).sum())
            fila[f"{grupo}: variazione media"] = round(float(cambio.mean()), 2)
        filas.append(fila)
    lex = LEXICO[idioma]
    fila = {"lingua": idioma, "approccio": "(d) lessico con finestra"}
    for grupo, pares, signo in (("POS", PARES_POS, -1), ("NEG", PARES_NEG, +1)):
        cambio = np.array([puntuar(n, lex, idioma) - puntuar(b, lex, idioma) for b, n in pares[idioma]])
        assert (signo * cambio > 0).all()          # proprietà dell'analizzatore: inverte tutte le dodici coppie
        fila[f"{grupo}: coppie ben orientate"] = int((signo * cambio > 0).sum())
        fila[f"{grupo}: variazione media"] = round(float(cambio.mean()), 2)
    filas.append(fila)
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** Il gruppo `POS` («non è buono») è il caso «facile»: i modelli supervisionati lo orientano bene **perché in questo corpus `non` compare soprattutto in recensioni negative e ha imparato un peso negativo**, non perché «capiscano» la negazione. Il gruppo `NEG` («non è cattivo») lo smaschera: aggiungere `non` **continua ad abbassare** il margine nei tre approcci supervisionati (0 su 6 ben orientate) e la variazione media è quasi la stessa di `POS`: l'effetto di `non` è una costante, indipendente dalla parola che nega. Le marcature `NEG_` qui non lo sistemano perché in un corpus così piccolo ci sono quasi zero esempi di `NEG_cattivo`. Il lessico con finestra **inverte per costruzione** (lo verifica l'`assert`): è una regola, non un apprendimento, e ha i suoi errori (sezione 3).

In [ ]:
filas = []
for idioma in IDIOMAS:
    y = test[idioma]["etiqueta"]
    for nombre, modelo in (("(a)", modelo_supervisado((1, 1))), ("(b)", modelo_supervisado((1, 2))), ("(c)", modelo_con_marcas(idioma))):
        modelo.fit(train[idioma]["texto"], train[idioma]["etiqueta"])
        pred = modelo.predict(test[idioma]["texto"])
        filas.append({"lingua": idioma, "approccio": nombre, "accuratezza sul test": round(accuracy_score(y, pred), 3)})
print(pd.DataFrame(filas).to_string(index=False))

Da (a) a (c) l'accuratezza sale di qualche punto (≈ 5 recensioni su 90 in ciascuna lingua), nella direzione attesa, ma con 90 recensioni di test **è dentro il rumore** della sezione 6, e la negazione riguarda poche recensioni (nella sezione 8 vedrai che solo ≈ 8 % delle recensioni in spagnolo e ≈ 3 % in italiano hanno un'inversione applicata dall'analizzatore). È un esempio del perché conviene guardare **casi concreti** oltre alla media.

---
## 8. Analisi degli errori per categorie

Una sola percentuale nasconde **dove** sbaglia ciascun metodo. Costruiamo una tabella con le previsioni delle 300 recensioni di ciascuna lingua: il lessico (soglia fissata con l'addestramento) e il modello supervisionato, quest'ultimo con **validazione incrociata a 5 partizioni** perché nessuna previsione venga da un modello che ha visto quella recensione. Definiamo le categorie **guardando solo il testo**, dopo aver ispezionato gli errori nelle prime prove:

- `senza_lessico`: nessuna parola del testo è nel lessico;
- `con_negazione`: qualche parola è stata invertita da una negazione;
- `misto`: ci sono parole positive e negative;
- `polarità_chiara`: solo parole di un segno, senza negazione.

Usiamo inoltre la colonna `dificil` e la **classe reale**.

In [ ]:
CAT_SIN = "senza_lessico"      # nessuna parola del testo è nel lessico
CAT_NEG = "con_negazione"    # qualche parola del lessico è stata invertita da una negazione
CAT_MIX = "misto"                   # ci sono parole positive e negative
CAT_CLARA = "polarità_chiara"   # solo parole di un segno, senza negazione
ORDEN_CAT = [CAT_SIN, CAT_NEG, CAT_MIX, CAT_CLARA]

def categoria(texto, lexico, idioma):
    """Categoria del testo secondo ciò che vede il lessico (guarda solo il testo, non l'etichetta reale)."""
    _, aportes = analizar(texto, lexico, idioma)
    if not aportes:
        return CAT_SIN
    if any(base * final < 0 for _, base, final in aportes):
        return CAT_NEG
    if len({np.sign(final) for _, _, final in aportes}) > 1:
        return CAT_MIX
    return CAT_CLARA

In [ ]:
def tabla_predicciones(idioma):
    df = resenas[idioma].reset_index(drop=True).copy()
    lex = LEXICO[idioma]
    df["puntuacion"] = [puntuar(t, lex, idioma) for t in df["texto"]]
    df["pred_lex"] = [clasificar(p, UMBRAL[idioma]) for p in df["puntuacion"]]
    validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    df["pred_lr"] = cross_val_predict(modelo_supervisado(), df["texto"], df["etiqueta"], cv=validacion)
    df["categoria"] = [categoria(t, lex, idioma) for t in df["texto"]]
    df["ok_lex"] = df["pred_lex"] == df["etiqueta"]
    df["ok_lr"] = df["pred_lr"] == df["etiqueta"]
    return df

PRED = {idioma: tabla_predicciones(idioma) for idioma in IDIOMAS}
for idioma, df in PRED.items():
    print(f"{idioma}: accuratezza sulle 300 recensioni · lessico {df['ok_lex'].mean():.3f} · TF-IDF + LR (CV5) {df['ok_lr'].mean():.3f}")

In [ ]:
for idioma, df in PRED.items():
    por_cat = df.groupby("categoria").agg(n=("ok_lex", "size"), acierto_lex=("ok_lex", "mean"), acierto_lr=("ok_lr", "mean")).round(3)
    por_cat = por_cat.reindex([c for c in ORDEN_CAT if c in por_cat.index])
    por_dif = df.groupby("dificil").agg(n=("ok_lex", "size"), acierto_lex=("ok_lex", "mean"), acierto_lr=("ok_lr", "mean")).round(3)
    por_clase = df.groupby("etiqueta").agg(n=("ok_lex", "size"), acierto_lex=("ok_lex", "mean"), acierto_lr=("ok_lr", "mean")).round(3)
    print(f"=== {idioma}: per categoria del testo ===");  print(por_cat)
    print(f"--- {idioma}: per la colonna dificil ---");  print(por_dif)
    print(f"--- {idioma}: per classe reale ---");  print(por_clase)
    print()

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for eje, idioma in zip(ejes, IDIOMAS):
    df = PRED[idioma]
    tab = df.groupby("categoria")[["ok_lex", "ok_lr"]].mean().reindex([c for c in ORDEN_CAT if c in set(df["categoria"])])
    x = np.arange(len(tab))
    eje.bar(x - 0.2, 1 - tab["ok_lex"], width=0.4, label="lessico", color="#8fb4d9")
    eje.bar(x + 0.2, 1 - tab["ok_lr"], width=0.4, label="TF-IDF + LR", color="#e9a15b")
    eje.set_xticks(x)
    eje.set_xticklabels([f"{c}\n(n={int((df['categoria'] == c).sum())})" for c in tab.index], fontsize=7)
    eje.set_title(idioma)
ejes[0].set_ylabel("tasso di errore")
ejes[0].legend()
plt.suptitle("Errore per categoria del testo (300 recensioni)")
plt.tight_layout()
plt.show()

**Cosa osservi?** Guarda dove si concentra l'errore di ciascun metodo. Quando **il lessico non ha nessuna parola** della recensione (`senza_lessico`), può solo dire «neutro»: azzecca le neutre e sbaglia le positive e negative che usano parole che non conosce. E osserva la riga della **classe reale**: il `neutro` è la classe più difficile per il lessico, perché «carino ma piccolo» somma un +1 e un −1 e il risultato dipende da quanto pesa ciascuna parola.

### Leggere gli errori (la parte che non si può automatizzare)

Le tabelle dicono *quanto*; per sapere *perché* bisogna **leggere**. Stampiamo pochi errori del lessico in ciascuna categoria con il punteggio ricevuto. Ricorda: qualunque idea che ne esca (aggiungere parole al lessico, cambiare una soglia) va convalidata su **dati nuovi**; se ritocchi il lessico guardando questi errori e misuri di nuovo sulle stesse recensioni, la cifra non è più onesta.

In [ ]:
for idioma, df in PRED.items():
    errores = df[~df["ok_lex"]].sort_values("id")
    print(f"=== {idioma}: {len(errores)} errori del lessico ===")
    for cat in ORDEN_CAT:
        sub = errores[errores["categoria"] == cat]
        print(f"--- {cat}: {len(sub)} errori")
        for _, f in sub.head(3).iterrows():
            print(f"   [{f['etiqueta']} → {f['pred_lex']}, {f['puntuacion']:+.1f}, dificil={f['dificil']}] {f['texto_limpio'][:120]}")
    print()

---
## 9. Il sarcasmo e l'ironia: un limite che né il lessico né il sacco di parole risolvono

In «Che meraviglia di stampante: in un mese si è inceppata sette volte» le parole sono positive e **l'opinione è negativa**. Nessun lessico e nessun sacco di parole vede il contrasto con il mondo che rende ironica la frase. Cerchiamo i casi in cui il lessico prevede `positivo` per una recensione realmente `negativo` (candidati a ironia, ma **non lo sono tutti**: li leggeremo).

In [ ]:
for idioma, df in PRED.items():
    candidatos = df[(df["etiqueta"] == "negativo") & (df["pred_lex"] == "positivo")].sort_values("id")
    print(f"=== {idioma}: {len(candidatos)} candidati (negative che il lessico vede come positive) · dificil=1: {int(candidatos['dificil'].sum())} · il modello supervisionato azzecca {int(candidatos['ok_lr'].sum())}/{len(candidatos)} ===")
    for _, f in candidatos.head(8).iterrows():
        print(f"   [dificil={f['dificil']}, {f['puntuacion']:+.1f}] {f['texto_limpio'][:125]}")
    print()

**Cosa osservi?** Leggi i candidati. Alcuni sono ironia o sarcasmo genuini («Bellissima la vista sul parcheggio…»); altri sono negativi per un'altra ragione (una lode isolata dentro una lamentela, una parola del lessico usata in un altro senso). Né il lessico né il modello hanno modo di distinguerli: le caratteristiche sono parole, e l'ironia sta nella **relazione tra ciò che si dice e ciò che ci si aspetta**. Riconoscerla richiede contesto, conoscenza del mondo o un modello molto più grande, e anche così sbaglia.

---
## 10. Riepilogo e passo successivo

- Il **lessico di polarità** è trasparente (ogni decisione si spiega parola per parola) e non richiede dati etichettati, ma la sua copertura è limitata e la sua qualità dipende dal lessico e dal dominio. Il nostro è minuscolo e da gioco.
- La **negazione** si tratta con una finestra di parole (con i suoi limiti: distanza, ambito) e gli **intensificatori** con moltiplicatori; entrambi sono regole scritte a mano.
- Il classificatore **supervisionato** impara dal corpus ciò che il lessico non ha, ma **richiede etichette** ed eredita i limiti del sacco di parole (la negazione si perde a meno che non venga codificata).
- **Confronta sempre sulla stessa partizione** e sapendo quanto rumore c'è nella cifra; con corpora piccoli, una differenza di pochi punti non è conclusiva.
- Una sola percentuale nasconde dove sbaglia il metodo: analizza gli errori **per categorie** e **leggili**.
- L'**ironia** e il sarcasmo restano fuori dalla portata di questi metodi.

**Prossimo passo:** nella lezione 7, *Analisi grammaticale ed entità*, passiamo dal classificare testi interi all'analizzare la struttura di ogni frase con spaCy (POS, dipendenze e NER). Prima, fai il notebook di pratica di questa lezione.